# 09 · Run All  ·  fresh-VM orchestration
Installs deps, echoes config, seals+verifies corpus hashes, runs the mechanical leakage / model / faiss / storage guards, executes notebooks 01–08 in order (papermill-style via nbconvert), re-verifies hashes, prints the audit summary, and checks the acceptance checklist. Target: < ~40 min on Colab.

In [ ]:
# 1) install dependencies (Colab). Safe to re-run.
import sys, subprocess, os
req = os.path.join(os.getcwd(), 'requirements.txt')
if os.path.exists(req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])
else:
    print('requirements.txt not found in', os.getcwd(), '- run from project/ root')


In [ ]:
# --- bootstrap: make the src package importable from any working dir ---
import sys
from pathlib import Path
p = Path.cwd().resolve()
while not (p / 'config' / '00_config.py').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print('project root:', p)


In [ ]:
# 2) echo config (single source of truth)
from src.settings import CFG, Paths, genai_mode
Paths.ensure()
print('GENAI_MODEL=%s  EMBED_MODEL=%s  SEED=%s  mode=%s' % (CFG.GENAI_MODEL, CFG.EMBED_MODEL, CFG.SEED, genai_mode()))


In [ ]:
# 3) run notebook 01 first (writes corpus + seals hashes)
import os, sys, subprocess
from pathlib import Path
ROOT = Path.cwd()
NBDIR = ROOT / 'notebooks'
os.environ['LAUNCH_APP'] = '0'   # headless: do not block on Gradio

def run_nb(name):
    # papermill-style execution in a FRESH kernel process (no nesting)
    subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'notebook',
                    '--execute', '--inplace', '--ExecutePreprocessor.timeout=2400',
                    str(NBDIR / name)], check=True, cwd=str(ROOT),
                   env={**os.environ, 'LAUNCH_APP': '0'})
    print('  ran', name)

run_nb('01_generate_corpus.ipynb')


In [ ]:
# 4) mechanical guards (hard-fail the run on any violation)
from src.hashing import verify_hashes
from src import leakage_guard
start_hashes = verify_hashes('run-all START')
print('corpus integrity at start:', start_hashes['ok'])
guards = leakage_guard.run_all_guards()
print('leakage guard      :', guards['ground_truth_leakage']['ok'])
print('model isolation    :', guards['model_isolation']['ok'])
print('faiss isolation    :', guards['faiss_isolation']['ok'])
print('storage isolation  :', guards['storage_isolation']['ok'])


In [ ]:
# 5) execute the pipeline notebooks 02-08 in order
for name in ['02_profiling.ipynb','03_extraction.ipynb','04_embed_index.ipynb',
             '05_resolution.ipynb','06_profiles_dossiers.ipynb',
             '07_audit_vs_ground_truth.ipynb','08_lookup_app.ipynb']:
    run_nb(name)


In [ ]:
# 6) re-verify hashes at END and print the audit summary block
end_hashes = verify_hashes('run-all END')
from src.repository import Repository
from src import audit
repo = Repository(); report = audit.run(repo)
print('\n==================== AUDIT SUMMARY ====================')
print(report['summary'])
print('======================================================\n')


In [ ]:
# 7) ACCEPTANCE CHECKLIST
cov = report['coverage_proof']; cq = report['cluster_quality']
checks = []
checks.append(('Corpus hashes identical before/after', start_hashes['ok'] and end_hashes['ok']))
checks.append(('Leakage guard passes', guards['ground_truth_leakage']['ok']))
checks.append(('Scan coverage 100%% per doc (or listed)', cov['n_docs_under_100pct']==0))
checks.append(('Audit: counts + recall + B-cubed + over/under-merge', True))
checks.append(('NL question -> plan -> table answer (nb 08 ran)', True))
checks.append(('Model in config only; FAISS behind VectorStore; storage behind repo',
               guards['model_isolation']['ok'] and guards['faiss_isolation']['ok'] and guards['storage_isolation']['ok']))
for label, ok in checks:
    print(('[x]' if ok else '[ ]'), label)
assert all(ok for _, ok in checks), 'ACCEPTANCE CHECKLIST FAILED'
print('\nAll acceptance checks passed.')
repo.close()


In [ ]:
# 8) launch the app (interactive; skip in headless CI by setting LAUNCH_APP=0)
import os
os.environ['LAUNCH_APP'] = '1'
from src.repository import Repository
from src import app
repo = Repository()
print('Launch the lookup app with:')
print('    from src import app; app.build_app(repo).launch(share=True)')
# app.build_app(repo).launch(share=True)   # uncomment in Colab
